# 10 Frozen Independent Test and Synthesis

**Scientific question.** Is independent evaluation possible under the frozen contract?

**Learning objectives.** Check hashes and score eligibility without selecting on test data.

**Prerequisites.** Install the project requirements. Configure the data directory for raw-record checks. Portable evidence review also uses the existing run artifacts.

**Dataset and run.** Course gearbox at 3.2 kHz, the project’s designated training and held-out development records, and the explicit diagnostic or blind availability files where relevant.

**Execution.** This clean-kernel replay reads the evidence cited in each code cell. Use the explicit project pipeline for full fitting and the notebook build script for review replay. The companion guide contains notation, figures, and worked examples.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
get_ipython().run_line_magic("matplotlib", "inline")
from IPython.display import Image, display

hint = os.environ.get("SRC_PROJECT_DIR")
locations = [Path(hint).resolve()] if hint else [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in locations if (p / "src" / "src_gearbox").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open within the project or set SRC_PROJECT_DIR.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
DATA = Path(os.environ.get("SRC_DATA_DIR", ROOT / "data" / "raw"))
RESULTS = ROOT / "results"
START = 1792
N = 24000
pd.options.display.max_colwidth = 120
pd.options.display.width = 115
plt.rcParams.update({"figure.dpi": 105, "axes.grid": True, "grid.alpha": .18})

def table(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return None
    return pd.read_csv(file)

def evidence(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return {}
    return json.loads(file.read_text(encoding="utf-8"))

def show_fig(relative, width=780):
    file = ROOT / relative
    if file.is_file():
        display(Image(filename=str(file), width=width))
    else:
        print(f"NOT AVAILABLE YET · {relative}")

def forecast(relative):
    file = ROOT / relative
    if not file.is_file():
        print(f"NOT AVAILABLE YET · {relative}")
        return {}
    with np.load(file, allow_pickle=False) as source:
        arrays = {key: source[key] for key in source.files}
    np.testing.assert_array_equal(arrays["target_indices"], np.arange(START, N))
    assert np.isfinite(arrays["actual"]).all()
    return arrays

def align_forecasts(reference, *others):
    for other in others:
        for key in ("target_indices", "actual", "rpm", "time_s"):
            np.testing.assert_array_equal(reference[key], other[key])

def draw_forecast(bundle, keys, view=slice(5000, 5200)):
    if not bundle:
        return
    y, t = bundle["actual"], bundle["time_s"]
    fig, ax = plt.subplots(2, 1, figsize=(9.0, 4.0), sharex=True, layout="constrained")
    ax[0].plot(t[view], y[view], color="#141a26", lw=1.2, label="Measured")
    for key, label in keys.items():
        if key in bundle:
            pred = bundle[key]
            ax[0].plot(t[view], pred[view], lw=0.9, label=label)
            ax[1].plot(t[view], (y-pred)[view], lw=0.9, label=label)
    ax[0].legend(ncol=3, fontsize=8)
    ax[0].set_ylabel("Std. vibration")
    ax[1].set(ylabel="Std. residual", xlabel="Acquisition time (s)")
    plt.show()

print("Read-only evidence replay.", f"Python {platform.python_version()}",
      f"NumPy {np.__version__}. One-step observed history through t−1.")

Read-only evidence replay. Python 3.12.14 NumPy 2.5.5. One-step observed history through t−1.


### Hashes first, test availability second

The selected configuration is a result of training-only selection. The
one-time independent route checks hashes and keeps settings frozen before
reading any newly supplied test record. If blind files are unavailable,
the notebook preserves that status and avoids creating scores. This is a
read-only checkpoint summary. Explicit testing belongs to the independent
evaluation script after inspecting availability and the frozen manifest.

Mid-record predictions use measured histories through $t-1$ and a common
warm-up. A confirmed development improvement demonstrates performance on
this segment and gives no independent-fault estimate without suitable
inputs and ground truth. Companion Chapters 20, 22–25 document the checks.

In [2]:
from src_gearbox.data import find_input
from src_gearbox.data.io_matlab import fault_sources

blind = {"sigtest":find_input(DATA,"sigtest") is not None,
         "sigfault2":bool(fault_sources(DATA,"sigfault2"))}
inventory = evidence("results/data_availability.json")
display(pd.DataFrame({
   "record":list(blind),
   "local input status":["PRESENT" if value else "NOT AVAILABLE" for value in blind.values()]
}))
if not any(blind.values()):
    print("Independent evaluation blocked by missing inputs.")
print("Recorded acquisition-label status:",inventory.get("label_status","NOT AVAILABLE"))

,record,local input status
0,sigtest,NOT AVAILABLE
1,sigfault2,NOT AVAILABLE


Independent evaluation blocked by missing inputs.
Recorded acquisition-label status: A/B are index groups only; no authoritative health labels supplied


In [3]:
registry = evidence("checkpoints/src_registry.json")
if registry.get("models"):
    primary = next(item for item in registry["models"] if item["model"]=="primary_pi_src")
    assert hashlib.sha256((ROOT/primary["checkpoint"]).read_bytes()).hexdigest() == primary["sha256"]
    display({"primary":primary["model"],"checkpoint_sha256":primary["sha256"],
             "training_sha256":registry["training_sha256"],
             "mask_sha256":registry["target_mask_sha256"]})
for record in ("availability.json","fault2_availability.json"):
    data = evidence(f"results/frozen/{record}")
    if data:
        display(data)

{'primary': 'primary_pi_src',
 'checkpoint_sha256': '488371b5be57102483896c77adc3f4182a98f2d477a9d201036eea93f0875014',
 'training_sha256': '47d23efd20d86ed9e67181a96c9b73039a79a874e0d9ae91be5bcb8a421c19a6',
 'mask_sha256': '1522c0a1b184b971b59763df3d827515370162751cc61209f025765039a89fb1'}

{'status': 'NOT AVAILABLE',
 'target': 'sigtest',
 'checkpoint_sha256': '488371b5be57102483896c77adc3f4182a98f2d477a9d201036eea93f0875014',
 'split': 'frozen independent input',
 'fault2_available': False,
 'verified_labels_supplied': False,
 'normalizers_refit': False,
 'reason': 'No independent vibration test was supplied.'}

{'status': 'NOT AVAILABLE',
 'dataset': 'sigfault2',
 'reason': 'No blind fault records supplied'}

### Interpretation and limits

A freeze fixes the variant, adapter, transforms, and diagnostic policy.
Loading a checkpoint and displaying availability consumes no test
targets. The independent route should evaluate once under that contract.
The course development predictions and provisional detection groups are
scientifically meaningful only with their stated scope.

Read the [Companion Guide](../docs/companion_guide.md) and the [Project README](../README.md).